<a href="https://colab.research.google.com/github/maxGrigorenko/MP-SENet_modification/blob/main/MP_SENet_Mamba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Сравнение скорости и качества обучения на первых эпохах архитектуры MP-SENet с оригинальными блоками TS-Conformer и Mamba

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124

!pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl

!pip install https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl

!pip install packaging triton timm==0.4.12 pytest chardet yacs termcolor submitit tensorboardX fvcore seaborn opencv-python tensorboard
!pip install transformers==4.40.0

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 512.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 25.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB ? 

In [ ]:
!git clone https://github.com/DmitryRyumin/MP-SENet.git
%cd MP-SENet

# Удаляем жесткие привязки к версиям в requirements.txt
!sed -i 's/==.*//g' requirements.txt

!pip install -r requirements.txt

Cloning into 'MP-SENet'...
remote: Enumerating objects: 903, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 903 (delta 166), reused 140 (delta 140), pack-reused 716 (from 2)
Receiving objects: 100% (903/903), 509.78 MiB | 26.20 MiB/s, done.
Resolving deltas: 100% (247/247), done.
Updating files: 100% (250/250), done.
/content/MP-SENet
  Preparing metadata (setup.py) ... done
  Created wheel for pesq: filename=pesq-0.0.4-cp312-cp312-linux_x86_64.whl size=284117 sha256=bd2d44dd60334acb3b01fdae03639f49ddc00e2773dcb17b3b6ec34ea33103cb
  Stored in directory: /root/.cache/pip/wheels/9b/d4/a4/9cf3512534cd47ce4a036d1593ee4013f2bf7509e631a147a3
Successfully built pesq


In [ ]:
!pip install -q -U gdown

# Скачиваем папку напрямую по ID из README авторов
print("Скачиваем датасет с Google Drive (~2 Гб)...")
!gdown --folder 19I_thf6F396y5gZxLTxYIojZXC0Ywm8l -O VoiceBank+DEMAND

# Защита от "матрешки" (если gdown скачал папку в папку)
!if [ -d "VoiceBank+DEMAND/VoiceBank+DEMAND" ]; then \
    mv VoiceBank+DEMAND/VoiceBank+DEMAND/* VoiceBank+DEMAND/ && \
    rm -rf VoiceBank+DEMAND/VoiceBank+DEMAND; \
fi

# Проверяем, что нужные папки на месте
!ls -l VoiceBank+DEMAND
print("Датасет загружен")

Скачиваем датасет с Google Drive (~2 Гб)...
Retrieving folder contents
Processing file 1mcstALH68vf6h-Dk2cUVl7W9eLNgsm56 test.txt
Processing file 16Jhr7FNNaHhypPZQIjBvuIWMVvXRDZVn training.txt
Processing file 1MKTUu6E9GNUgdkHfJrB8fILZQoJWChXz wav_clean.zip
Processing file 15tYmw3h8C7j2QPaMflMocH3pMzpuH-pp wav_noisy.zip
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1mcstALH68vf6h-Dk2cUVl7W9eLNgsm56
To: /content/MP-SENet/VoiceBank+DEMAND/test.txt
100% 40.4k/40.4k [00:00<00:00, 62.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=16Jhr7FNNaHhypPZQIjBvuIWMVvXRDZVn
To: /content/MP-SENet/VoiceBank+DEMAND/training.txt
100% 567k/567k [00:00<00:00, 108MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1MKTUu6E9GNUgdkHfJrB8fILZQoJWChXz
From (redirected): https://drive.google.com/uc?id=1MKTUu6E9GNUgdkHfJrB8fILZQoJWChXz&confirm=t&uuid=0259d970-08c1-4f0c-ba30-20f

In [ ]:
!mkdir -p VoiceBank+DEMAND/wavs_clean VoiceBank+DEMAND/wavs_noisy

print("Распаковываем чистые аудио...")
!unzip -q VoiceBank+DEMAND/wav_clean.zip -d VoiceBank+DEMAND/tmp_clean
# Гарантированно вытаскиваем все .wav файлы из любых подпапок в корень wavs_clean
!find VoiceBank+DEMAND/tmp_clean -name "*.wav" -exec mv {} VoiceBank+DEMAND/wavs_clean/ \;

print("Распаковываем зашумленные аудио...")
!unzip -q VoiceBank+DEMAND/wav_noisy.zip -d VoiceBank+DEMAND/tmp_noisy
# То же самое для зашумленных
!find VoiceBank+DEMAND/tmp_noisy -name "*.wav" -exec mv {} VoiceBank+DEMAND/wavs_noisy/ \;

# Убираем мусор и временные папки
!rm -rf VoiceBank+DEMAND/tmp_clean VoiceBank+DEMAND/tmp_noisy VoiceBank+DEMAND/*.zip

Распаковываем чистые аудио...
Распаковываем зашумленные аудио...


In [ ]:
import os
import random
import shutil

clean_dir = 'VoiceBank+DEMAND/wavs_clean'
noisy_dir = 'VoiceBank+DEMAND/wavs_noisy'

# Получаем все файлы и перемешиваем
files = sorted(os.listdir(clean_dir)) # Сортируем для стабильности, затем шафлим
random.seed(42)
random.shuffle(files)

keep_count = int(len(files) * 0.05)
files_to_remove = files[keep_count:]

# Создаем папки для бэкапа
os.makedirs('VoiceBank+DEMAND/backup_clean', exist_ok=True)
os.makedirs('VoiceBank+DEMAND/backup_noisy', exist_ok=True)

print(f"Всего файлов: {len(files)}. Оставляем 5% ({keep_count} файлов)...")

# Убираем лишнее в бэкап
for f in files_to_remove:
    # Используем try-except на случай, если скрипт запускается дважды
    try:
        shutil.move(os.path.join(clean_dir, f), os.path.join('VoiceBank+DEMAND/backup_clean', f))
        shutil.move(os.path.join(noisy_dir, f), os.path.join('VoiceBank+DEMAND/backup_noisy', f))
    except FileNotFoundError:
        pass

Всего файлов: 12397. Оставляем 5% (619 файлов)...


In [ ]:
clean_dir = 'VoiceBank+DEMAND/wavs_clean'
valid_names = sorted([f.replace('.wav', '') for f in os.listdir(clean_dir) if f.endswith('.wav')])

random.seed(42)
random.shuffle(valid_names)

split_idx = int(len(valid_names) * 0.9)
train_names = valid_names[:split_idx]
test_names = valid_names[split_idx:]

with open('VoiceBank+DEMAND/training.txt', 'w') as f:
    f.write('\n'.join(train_names))

with open('VoiceBank+DEMAND/test.txt', 'w') as f:
    f.write('\n'.join(test_names))

print("Списки сгенерированы")
print(f"В training.txt записано: {len(train_names)} файлов.")
print(f"В test.txt записано: {len(test_names)} файлов.")

Списки сгенерированы
В training.txt записано: 556 файлов.
В test.txt записано: 62 файлов.


In [ ]:
file_path = 'models/transformer.py'

with open(file_path, 'r') as f:
    content = f.read()

# Меняем строку (авторы указали на баг и рекмендовали добавить batch_first=True)
old_str = "self.attention = MultiheadAttention(d_model, n_heads, dropout=dropout)"
new_str = "self.attention = MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)"

# Делаем замену и перезаписываем файл
if old_str in content:
    content = content.replace(old_str, new_str)
    with open(file_path, 'w') as f:
        f.write(content)
    print("Успех")
else:
    print("Строка не найдена")

# Выведем строчку, чтобы убедиться
!grep "MultiheadAttention" models/transformer.py

Успех
from torch.nn import MultiheadAttention, GRU, Linear, LayerNorm, Dropout
        self.attention = MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)


In [ ]:
%%writefile models/mamba_net.py
import torch
import torch.nn as nn
from mamba_ssm import Mamba

# Импортируем все вспомогательные функции и оригинальный класс для наследования
from models.model import *
from models.model import MPNet as OriginalMPNet

class MambaTimeFreqBlock(nn.Module):
    """
    Блок двумерного сканирования спектрограммы на базе State Space Моделей (Mamba).
    Последовательно обрабатывает временные зависимости (Time) и частотные (Frequency),
    используя Pre-LayerNorm и остаточные связи.
    """
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        # Нормализация перед блоками (Pre-LN) для стабилизации обучения
        self.norm1 = nn.LayerNorm(d_model)
        self.time_mamba = Mamba(
            d_model=d_model,    # Соответствует h.dense_channel (64)
            d_state=d_state,    # Размерность скрытого состояния SSM
            d_conv=d_conv,      # Размер ядра локальной 1D-свертки внутри Mamba
            expand=expand       # Фактор расширения каналов внутри блока
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.freq_mamba = Mamba(
            d_model=d_model,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand
        )

    def forward(self, x):
        # Входной тензор от DenseEncoder имеет размерность: [B, C, T, F]
        # B - Batch size, C - Channels (Embedding), T - Time frames, F - Frequency bins
        b, c, t, f = x.size()

        # -----------------------------------------------------------------
        # 1. Сканирование по оси Времени (Time-Scanning)
        # -----------------------------------------------------------------
        res = x
        # Переносим ось времени на место длины последовательности L
        x_t = x.permute(0, 3, 2, 1).contiguous()  # [B, F, T, C]
        x_t = self.norm1(x_t)
        # Схлопываем батч и частоты: Mamba обрабатывает каждый частотный бин независимо вдоль времени
        x_t = x_t.view(b * f, t, c)               # [B * F, L=T, D=C]
        x_t = self.time_mamba(x_t)
        # Возвращаем исходную геометрию тензора
        x_t = x_t.view(b, f, t, c).permute(0, 3, 2, 1).contiguous()  # [B, C, T, F]
        x = res + x_t                             # Residual connection

        # -----------------------------------------------------------------
        # 2. Сканирование по оси Частоты (Frequency-Scanning)
        # -----------------------------------------------------------------
        res = x
        # Переносим ось частот на место длины последовательности L
        x_f = x.permute(0, 2, 3, 1).contiguous()  # [B, T, F, C]
        x_f = self.norm2(x_f)
        # Схлопываем батч и временные фреймы: Mamba обрабатывает каждый момент времени вдоль частот
        x_f = x_f.view(b * t, f, c)               # [B * T, L=F, D=C]
        x_f = self.freq_mamba(x_f)
        # Возвращаем исходную геометрию тензора
        x_f = x_f.view(b, t, f, c).permute(0, 3, 1, 2).contiguous()  # [B, C, T, F]
        x = res + x_f                             # Residual connection

        return x

class MPNetMamba(OriginalMPNet):
    """
    Модифицированная архитектура MP-SENet, где блоки трансформера
    заменены на двумерные блоки Mamba SSM.
    """
    def __init__(self, h, num_tsblocks=4):
        super().__init__(h, num_tsblocks)

        # Извлекаем специфичные для Mamba параметры из конфигурации или берем дефолты
        d_state = getattr(h, 'd_state', 16)
        d_conv = getattr(h, 'd_conv', 4)
        expand = getattr(h, 'expand', 2)

        # Полная замена оригинального стека TSTransformerBlock
        self.TSTransformer = nn.ModuleList([
            MambaTimeFreqBlock(
                d_model=h.dense_channel,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand
            ) for _ in range(num_tsblocks)
        ])

# Переопределяем имя класса для бесшовной подмены импорта в train.py
MPNet = MPNetMamba

Writing models/mamba_net.py


In [ ]:
%%writefile run_mamba_search.py
import json
import random
import subprocess
import copy
import sys
import os
import signal

def patch_train_script():
    with open('train.py', 'r') as f:
        content = f.read()
    content = content.replace('models.tf_mixer_net', 'models.model')
    content = content.replace('models.mamba_net', 'models.model')
    content = content.replace('MPNetMixer as MPNet', 'MPNet')
    content = content.replace('from models.model import', 'from models.mamba_net import')
    with open('train.py', 'w') as f:
        f.write(content)

def generate_random_config(base_config_path):
    with open(base_config_path, 'r') as f:
        config = json.load(f)

    mc_config = copy.deepcopy(config)
    mc_config['batch_size'] = 2
    mc_config['num_workers'] = 0

    # Случайный перебор параметров
    mc_config['learning_rate'] = random.uniform(1e-4, 6e-4)
    mc_config['d_state'] = random.choice([8, 16, 32])
    mc_config['expand'] = random.choice([1, 2])
    mc_config['d_conv'] = 4

    return mc_config

def run_experiment(config_path, max_epochs=3):
    env = os.environ.copy()
    env["LOKY_MAX_CPU_CORES"] = "1"
    env["OMP_NUM_THREADS"] = "1"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    cmd = f"python train.py --config {config_path}"
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)

    epoch_losses = []
    best_epoch_avg = float('inf')
    current_epoch = 1

    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()

        if "Gen Loss:" in line:
            try:
                loss_val = float(line.split("Gen Loss:")[1].split(",")[0].strip())
                epoch_losses.append(loss_val)
            except Exception:
                pass

        if f"Time taken for epoch" in line or f"Epoch: {current_epoch + 1}" in line:
            if epoch_losses:
                epoch_avg = sum(epoch_losses) / len(epoch_losses)
                print(f"\n[Аналитика] Эпоха {current_epoch} завершена. Средний Gen Loss: {epoch_avg:.4f}")
                if epoch_avg < best_epoch_avg:
                    best_epoch_avg = epoch_avg

            epoch_losses = []
            current_epoch += 1

            if current_epoch > max_epochs:
                print(f"\nДостигнут лимит эпох ({max_epochs}). Мягко останавливаем процесс (SIGINT)...")
                process.send_signal(signal.SIGINT)
                try:
                    process.wait(timeout=15)
                    print("Процесс корректно завершил работу.")
                except subprocess.TimeoutExpired:
                    print("Процесс завис. Добиваем принудительно (SIGKILL)...")
                    process.kill()
                break

    process.wait()
    return best_epoch_avg

if __name__ == "__main__":
    patch_train_script()
    base_config = 'config.json'

    results = []
    NUM_EXPERIMENTS = 3

    print("\n" + "="*60)
    print("СТАРТ СРАВНИТЕЛЬНОГО ТЕСТИРОВАНИЯ MAMBA SSM")
    print("="*60)

    for i in range(NUM_EXPERIMENTS):
        print(f"\n\n{'='*20} ЭКСПЕРИМЕНТ #{i+1} {'='*20}")
        config = generate_random_config(base_config)
        config_name = f"config_mamba_{i}.json"

        with open(config_name, 'w') as f:
            json.dump(config, f, indent=4)

        print(f"Параметры: LR={config['learning_rate']:.5f}, d_state={config['d_state']}, expand={config['expand']}")

        avg_loss = run_experiment(config_name, max_epochs=3)
        print(f"Итог эксперимента #{i+1}: СРЕДНИЙ Gen Loss = {avg_loss:.4f}")

        results.append({
            "exp": i + 1,
            "loss": avg_loss,
            "d_state": config['d_state'],
            "expand": config['expand'],
            "lr": config['learning_rate']
        })

    print("\n\n" + "ИТОГОВАЯ ТАБЛИЦА ЭКСПЕРИМЕНТОВ MAMBA")
    print(f"{'Exp':<5} | {'Mean Gen Loss':<15} | {'d_state':<8} | {'expand':<8} | {'Learn Rate'}")
    print("-" * 65)

    results.sort(key=lambda x: x['loss'])

    for r in results:
        best_marker = "*" if r == results[0] else "  "
        print(f"{best_marker} {r['exp']:<2} | {r['loss']:<15.4f} | {r['d_state']:<8} | {r['expand']:<8} | {r['lr']:.5f}")

Writing run_mamba_search.py


In [ ]:
!rm -rf /dev/shm/*

In [ ]:
!python run_mamba_search.py


СТАРТ СРАВНИТЕЛЬНОГО ТЕСТИРОВАНИЯ MAMBA SSM


==================== ЭКСПЕРИМЕНТ #1 ====================
Параметры: LR=0.00055, d_state=16, expand=2
2026-06-21 21:27:00.497291: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Initializing Training Process..
Batch size per GPU : 2
MPNetMamba(
  (dense_encoder): DenseEncoder(
    (dense_conv_1): Sequential(
      (0): Conv2d(2, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (2): PReLU(num_parameters=64)
    )
    (dense_block): DenseBlock(
      (dense_block): ModuleList(
        (0): Sequential(
          (0): ConstantPad2d(padding=(1, 1, 1, 0), value=0.0)
          (1): Conv2d(64, 64, kernel_size=(2, 3), st

В ходе экспериментов на усеченной выборке (5%) был зафиксирован паттерн роста функции потерь Генератора (Gen Loss) на третьей эпохе, характерный для всех протестированных архитектур (Baseline, TFMixer, Mamba). Данное поведение является ожидаемым артефактом адверсариального обучения (GAN): быстрый выход Дискриминатора на плато переобучения приводит к ужесточению штрафов для Генератора, из-за чего значение Gen Loss перестает линейно коррелировать с фактическим качеством денойзинга.

Финальное сравнение (по Gen Loss):

Baseline (Conformer): ~0.9945. Дает лучшее качество, но невероятно прожорлив до памяти.


Mamba SSM: ~0.9993. Отличный компромисс. Практически догнала трансформер по качеству, избавившись от механизма внимания, но всё еще требует аккуратной работы с памятью.


TFMixer: ~1.0229. Экстремально легкая архитектура на 1D-свертках. Дала небольшую просадку по лоссу, но взамен обеспечила высочайшую скорость работы и стабильность.